In [ ]:
import pandas as pd

# 1. Load the individual CSV files
# Ensure the file paths match where you stored the extracted data
train = pd.read_csv('../data/train.csv')
stores = pd.read_csv('../data/stores.csv')
oil = pd.read_csv('../data/oil.csv')
holidays = pd.read_csv('../data/holidays_events.csv')
transactions = pd.read_csv('../data/transactions.csv')

# 2. Convert date columns to datetime objects for time-series operations
train['date'] = pd.to_datetime(train['date'])
oil['date'] = pd.to_datetime(oil['date'])
holidays['date'] = pd.to_datetime(holidays['date'])
transactions['date'] = pd.to_datetime(transactions['date'])

# 3. Merge the datasets incrementally using 'left' joins to keep all training data
# Add store metadata (city, state, type) using the shared 'store_nbr' column
master_df = pd.merge(train, stores, on='store_nbr', how='left')

# Add daily oil prices using the shared 'date' column
master_df = pd.merge(master_df, oil, on='date', how='left')

# Add daily transaction counts using both 'date' and 'store_nbr'
master_df = pd.merge(master_df, transactions, on=['date', 'store_nbr'], how='left')

# Keep only the first holiday recorded per date to avoid row duplication during merge
holidays_unique = holidays.drop_duplicates(subset=['date'], keep='first')

# Now it is safe to merge into the master dataframe
master_df = pd.merge(master_df, holidays_unique[['date', 'type', 'description']], on='date', how='left')

# Fill empty holiday rows (normal days) with a default string
master_df['type'] = master_df['type'].fillna('Normal_Day')

# (Optional) Rename the oil price column for clarity
master_df = master_df.rename(columns={'dcoilwtico': 'oil_price'})

# Display the first few rows of the fully merged dataset
display(master_df.head())